# D13 — YOLOv8 Full Dataset Training
## Deep Learning and VLM based Real-Time Fighting Detection on Beach Safety Management
**Student:** Singo Loua | 240086608 | MSc Applied Data Science  
**Supervisor:** Dr. Ming Jiang | University of Sunderland  
**Methodology:** CRISP-DM Phase 4 — Modelling  
**Date:** April 2026

---
### What this notebook does:
1. Connects to Google Drive
2. Unzips the full thermal silhouette dataset (60,438 frames)
3. Trains YOLOv8 nano classification on full 42,178 training images at 640px
4. Evaluates on the full 8,998-image test set
5. Downloads the best model weights back to your computer

**Expected training time:** 30-60 minutes on T4 GPU (vs 4+ days on CPU)


## STEP 1 — Check GPU is available

In [ ]:
# Check GPU
import torch

print('='*50)
print('GPU CHECK')
print('='*50)
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device     : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('STATUS: Ready for GPU training!')
else:
    print('WARNING: No GPU found!')
    print('Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU')
    print('Then run all cells again.')
print('='*50)

## STEP 2 — Install YOLOv8

In [ ]:
# Install Ultralytics YOLOv8
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')
print('YOLOv8 installed successfully!')

## STEP 3 — Connect Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted at /content/drive')

## STEP 4 — Unzip Dataset
> **Before running this cell:** Make sure you have uploaded `split_dataset.zip` to your Google Drive.  
> The zip file should be in your **My Drive** root folder.

In [ ]:
import os
import zipfile

# Path to your zip file on Google Drive
ZIP_PATH    = '/content/drive/MyDrive/split_dataset.zip'
EXTRACT_DIR = '/content/dataset'

# Check zip exists
if not os.path.exists(ZIP_PATH):
    print(f'ERROR: Zip file not found at {ZIP_PATH}')
    print('Please upload split_dataset.zip to your Google Drive root folder')
else:
    print(f'Found zip file: {ZIP_PATH}')
    print('Extracting dataset... (this may take a few minutes)')
    
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    
    print('Extraction complete!')
    print(f'Dataset location: {EXTRACT_DIR}')

# Verify dataset structure
DATASET_DIR = os.path.join(EXTRACT_DIR, 'split')
print('\nDataset structure:')
for split in ['train', 'val', 'test']:
    for cls in ['fight', 'no_fight']:
        folder = os.path.join(DATASET_DIR, split, cls)
        if os.path.exists(folder):
            count = len([f for f in os.listdir(folder) if f.endswith('.jpg')])
            print(f'  {split}/{cls}: {count:,} frames')
        else:
            print(f'  MISSING: {split}/{cls}')

## STEP 5 — Train YOLOv8 Full Dataset
> This is the main training cell. Expected time: **30-60 minutes** on T4 GPU.

In [ ]:
import torch
from ultralytics import YOLO
from datetime import datetime

print('='*60)
print('D13 YOLOv8 Full Dataset Training')
print('Student  : Singo Loua | 240086608')
print('Method   : CRISP-DM Phase 4 - Modelling')
print(f'Device   : {"GPU - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Started  : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print('='*60)

# Training configuration
DATASET_DIR = '/content/dataset/split'
EPOCHS      = 50
BATCH       = 32      # Larger batch with GPU
IMG_SIZE    = 640     # Full resolution matching D10 output
PATIENCE    = 15      # Early stopping
DEVICE      = 0 if torch.cuda.is_available() else 'cpu'

print(f'\nTraining config:')
print(f'  Dataset  : {DATASET_DIR}')
print(f'  Epochs   : {EPOCHS}')
print(f'  Batch    : {BATCH}')
print(f'  Img size : {IMG_SIZE}px')
print(f'  Patience : {PATIENCE}')
print(f'  Device   : {DEVICE}')
print('\nStarting training...\n')

# Load model
model = YOLO('yolov8n-cls.pt')

# Train
results = model.train(
    data     = DATASET_DIR,
    epochs   = EPOCHS,
    batch    = BATCH,
    imgsz    = IMG_SIZE,
    patience = PATIENCE,
    device   = DEVICE,
    project  = '/content/models',
    name     = 'd13_full',
    exist_ok = True,
    verbose  = True,
    plots    = True,
    save     = True,
    val      = True,
)

print('\n' + '='*60)
print('TRAINING COMPLETE')
print(f'Best model: /content/models/d13_full/weights/best.pt')
print('='*60)

## STEP 6 — Evaluate on Full Test Set

In [ ]:
from ultralytics import YOLO

print('Evaluating best model on full test set...')
best_model = YOLO('/content/models/d13_full/weights/best.pt')

metrics = best_model.val(
    data     = '/content/dataset/split',
    split    = 'test',
    device   = 0 if torch.cuda.is_available() else 'cpu',
    plots    = True,
    project  = '/content/models',
    name     = 'd13_test_eval',
    exist_ok = True,
)

print('\n' + '='*60)
print('D13 FINAL TEST RESULTS')
print('='*60)
print(f'Top-1 Accuracy : {metrics.top1*100:.2f}%')
print(f'Top-5 Accuracy : {metrics.top5*100:.2f}%')
print('='*60)
print(f'\nD12 baseline was: 75.33%')
print(f'D13 improvement : {(metrics.top1*100 - 75.33):.2f}%')

## STEP 7 — Save Results to Google Drive

In [ ]:
import shutil
import os

# Save model and results to Google Drive for download
DRIVE_OUTPUT = '/content/drive/MyDrive/beach_project_d13'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy model weights
shutil.copy(
    '/content/models/d13_full/weights/best.pt',
    f'{DRIVE_OUTPUT}/d13_best.pt'
)

# Copy results
shutil.copytree(
    '/content/models/d13_full',
    f'{DRIVE_OUTPUT}/d13_full',
    dirs_exist_ok=True
)

# Copy test evaluation
if os.path.exists('/content/models/d13_test_eval'):
    shutil.copytree(
        '/content/models/d13_test_eval',
        f'{DRIVE_OUTPUT}/d13_test_eval',
        dirs_exist_ok=True
    )

print('Results saved to Google Drive!')
print(f'Location: {DRIVE_OUTPUT}')
print('\nFiles saved:')
for f in os.listdir(DRIVE_OUTPUT):
    print(f'  {f}')
print('\nDownload d13_best.pt and copy to C:\\beach\\models\\ on your laptop')

## STEP 8 — Compare D12 vs D13 Results

In [ ]:
# Comparison table
print('='*65)
print('CRISP-DM MODELLING PHASE — RESULTS COMPARISON')
print('='*65)
print(f'{"Metric":<30} {"D12 Baseline":>15} {"D13 Full":>15}')
print('-'*65)
print(f'{"Training images":<30} {"5,000":>15} {"42,178":>15}')
print(f'{"Image resolution":<30} {"224px":>15} {"640px":>15}')
print(f'{"Epochs":<30} {"20":>15} {"50 (max)":>15}')
print(f'{"Batch size":<30} {"8":>15} {"32":>15}')
print(f'{"Hardware":<30} {"CPU":>15} {"GPU (T4)":>15}')
print(f'{"Training time":<30} {"1.07 hours":>15} {"~30-60 min":>15}')
print(f'{"Top-1 Accuracy":<30} {"75.33%":>15} {str(round(metrics.top1*100, 2))+"%":>15}')
print(f'{"Top-5 Accuracy":<30} {"100.00%":>15} {str(round(metrics.top5*100, 2))+"%":>15}')
print('='*65)
print('Next step: D14 - VLM integration (LLaVA:13b via Ollama)')